### Why this exists

A knowledge graph is really four problems, and only one of them wants an LLM:

| sub-problem | what we use |
|---|---|
| entity **mention** detection | AST for code, spaCy `noun_chunks` for prose, yake as fallback |
| entity **resolution** | usearch ANN + a lexical guard, plus shared-token blocking |
| **typed** relations | exact for code (`calls`/`imports`); PMI co-occurrence for prose |
| ontology design, NL querying over triples | *not covered — this is the LLM-shaped part* |

The bet is that a retrieval-serving graph needs edges that are **predictive of relevance**,
not edges that are **true statements**. Untyped co-occurrence edges are near-worthless as facts
and very useful as relatedness signal, which is all `graph_search` needs.

For code we do not guess at all: the AST already knows every symbol and every call, so NER
would only add error. spaCy is used on prose, where its `noun_chunks` (not its OntoNotes NER)
carry the domain terms.

In [ ]:
#| default_exp graph

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import Path, patch, merge, ifnone, first, L, AttrDict
from fastlite import Database
from apswutils.db import Table
import ast, math, re, zlib
import numpy as np

from litesearch.core import _in, _rid, _slug, _np_dtype, process_content

## Schema

Three tables beside the chunk store. Entities live in a regular `get_store` so they inherit the existing ANN machinery — resolution reuses `ann_search`/`rebuild_index` rather than adding a second index path.

In [ ]:
#| export
@patch
def get_graph(self:Database,
              store:str='store',   # chunk store the graph is built over
              prefix:str=None,     # table prefix (default: '' for 'store', else '<store>_')
              ann:bool=True,       # register an ANN index on the entity store (needed for resolution)
              **kw):               # extra typed columns for the entity store
    'Create the entity/mention/edge tables for a chunk store. Idempotent; returns the tables.'
    p = prefix if prefix is not None else ('' if store == 'store' else f'{store}_')
    en, mn, eg = f'{p}entities', f'{p}mentions', f'{p}edges'
    ents = self.get_store(en, hash=True, ann=ann, kind=str, canon=str, freq=int, **kw)
    self.t[mn].create(chunk_id=str, entity_id=str, surface=str, n=int,
                      pk=('chunk_id', 'entity_id'), if_not_exists=True)
    self.t[eg].create(src=str, dst=str, rel=str, weight=float, n=int,
                      pk=('src', 'dst', 'rel'), if_not_exists=True)
    for t, c in ((mn, 'entity_id'), (mn, 'chunk_id'), (eg, 'src'), (eg, 'dst')):
        self.t[t].create_index([c], if_not_exists=True)
    return AttrDict(entities=ents, mentions=self.t[mn], edges=self.t[eg], store=self.t[store], prefix=p)

## Normalisation and the lexical guard

`_lex_ok` is the gate on every proposed merge. Pure-embedding merging happily collapses `python 3.11` into `python 3.12`; requiring the numbers to agree, and requiring token containment to *cover* enough of the longer string, is what keeps the resolver honest.

In [ ]:
#| export
_DET = re.compile(r'^(the|a|an|this|that|these|those|its|their|our|your|his|her)\s+', re.I)
_WS  = re.compile(r'\s+')
_PRON = {'it','its','we','our','they','their','he','she','you','i','me','us','them','this','that',
         'these','those','which','who','what','there','here','one','ones','something','anything'}
_TOK = re.compile(r'[a-z0-9]+')
_NUM = re.compile(r'\d+')

def _norm(s):
    'Canonical surface form for a mention; None when the phrase is not entity-like.'
    if not s: return None
    s = _WS.sub(' ', s).strip().strip('.,;:!?()[]{}"\'`')
    s = _DET.sub('', s).strip()
    s = re.sub(r"'s$", '', s).strip()
    if not (2 <= len(s) <= 60): return None
    if len(s.split()) > 5: return None
    if s.lower() in _PRON: return None
    if not re.search(r'[A-Za-z]', s): return None          # pure numbers / punctuation
    return s.lower()

def _toks(s):
    '''Tokens for the lexical guard. UAX#29 treats `_` as a word joiner, so `fts_search` stays one
    token — splitting it gives {fts, search}, which merges it into `search` by containment.'''
    try: from apsw.unicode import word_iter, casefold
    except ImportError: return set(_TOK.findall(s.lower()))
    return {casefold(t) for t in word_iter(s or '') if any(c.isalnum() for c in t)}

def _acr(s):  return ''.join(w[0] for w in s.split() if w)

def _sentences(text):
    'UAX#29 sentence split via apsw; falls back to the whole text as one window.'
    try: from apsw.unicode import sentence_iter
    except ImportError: return [text]
    return [s for s in (x.strip() for x in sentence_iter(text)) if s]

def _jac(a, b):
    A, B = _toks(a), _toks(b)
    return len(A & B) / len(A | B) if (A | B) else 0.0

def _lex_ok(a, b, lex=0.34, cover=0.5):
    '''Lexical guard on a proposed merge.
    Blocks pairs whose numbers disagree ("python 3.11" vs "python 3.12"), and accepts token
    containment only when the shorter side covers `cover` of the longer — bare substring matching
    chains transitively through union-find and collapses the whole graph into one node.'''
    if a == b: return True
    if set(_NUM.findall(a)) != set(_NUM.findall(b)): return False
    if _acr(a) == b.lower() or _acr(b) == a.lower(): return True
    A, B = _toks(a), _toks(b)
    if not (A and B): return False
    inter = len(A & B)
    if inter == min(len(A), len(B)) and inter/max(len(A), len(B)) >= cover: return True
    return inter/len(A | B) >= lex

The guard, on the cases that actually bite:

In [ ]:
cases = [('python 3.11','python 3.12',False), ('usearch','usearch index',True),
         ('hierarchical navigable small world','hnsw',True), ('polonium','isolated polonium',True),
         ('curie','marie curie isolated',False), ('marie curie','marie curie isolated',True),
         ('attention','multi head attention',False), ('wmt 2014 dataset','the wmt 2014 dataset',True)]
for a,b,want in cases:
    assert _lex_ok(a,b) == want, (a,b,want)
print(f'{len(cases)} guard cases pass')

8 guard cases pass


Tokenisation decides whether that guard can work at all. `_toks` uses apsw's UAX#29
word segmentation, which treats `_` as a word joiner — so `fts_search` stays one token. Splitting it
into `{fts, search}` makes it a token-subset of `search` and the resolver merges the two by
containment, taking `vec_search` and `ann_search` with it.

The same choice matters in the store: FTS5's stock `porter unicode61` splits on `_` *and* stems the
pieces, so `fts_search` indexes as `ft`+`search`. `database()` registers apsw's tokenizers and
`get_store` defaults to `porter simplify casefold 1 unicodewords`, which keeps identifiers whole
while still stemming prose.

In [ ]:
import apsw, apsw.fts5
_c = apsw.Connection(':memory:'); apsw.fts5.register_tokenizers(_c, apsw.fts5.map_tokenizers)
for spec in (['porter','unicode61'], ['porter','simplify','casefold','1','unicodewords']):
    tk = _c.fts5_tokenizer(spec[0], spec[1:])
    got = [x[0] for x in tk(b'fts_search Running', apsw.FTS5_TOKENIZE_DOCUMENT, None, include_offsets=False)]
    print(f"{' '.join(spec):42s} -> {got}")

assert _toks('fts_search') == {'fts_search'}
assert not _lex_ok('fts_search', 'search'), 'identifiers must not collapse into their suffix'
print('\nidentifiers stay atomic')

porter unicode61                           -> ['ft', 'search', 'run']
porter simplify casefold 1 unicodewords    -> ['fts_search', 'run']

identifiers stay atomic


## A model-free embedder

Entity *names* are short strings where subword overlap is most of the signal, so a char-n-gram hash works well and needs no download. Chunk embeddings still want a real model — pass `FastEncode` for those.

In [ ]:
#| export
def hash_embed(texts,            # iterable of strings
               ndim:int=256,     # output dimensions
               ngram=(3,5),      # char n-gram range
               dtype=np.float16):
    '''Deterministic char-n-gram hashing embedder — no model, no download.
    Sized for *entity-name* resolution (subword overlap is what matters there) and for offline/CI runs.
    Use FastEncode for chunk embeddings, where you need real semantics.'''
    lo, hi = ngram
    txts = list(L(texts))
    out = np.zeros((len(txts), ndim), dtype=np.float32)
    for i, t in enumerate(txts):
        s = ' ' + _WS.sub(' ', (t or '').lower().strip()) + ' '
        for n in range(lo, hi+1):
            for j in range(len(s)-n+1):
                out[i, zlib.crc32(s[j:j+n].encode()) % ndim] += 1.0
    out /= np.clip(np.linalg.norm(out, axis=1, keepdims=True), 1e-12, None)
    return out.astype(dtype)

## Code entities — exact, from the AST

In [ ]:
#| export
_PY_SKIP = {'self','cls','super','print','len','str','int','float','bool','list','dict','set','tuple',
            'range','enumerate','zip','map','filter','isinstance','getattr','setattr','hasattr','type',
            'format','join','append','get','items','keys','values','open','sorted','sum','min','max'}

def _def_name(chunk):
    'Symbol defined by a code chunk, from pyparse metadata or the chunk source itself.'
    md = chunk.get('metadata') or {}
    if isinstance(md, dict) and md.get('name'): return md['name']
    try: tree = ast.parse(chunk['content'])
    except SyntaxError: return None
    n = first(tree.body, lambda x: isinstance(x, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return getattr(n, 'name', None)

def code_entities(chunk):
    'Exact symbols for a python chunk: (defined, called, imported). AST-derived, no model.'
    try: tree = ast.parse(chunk['content'])
    except (SyntaxError, ValueError): return None, L(), L()
    calls, imps = L(), L()
    for n in ast.walk(tree):
        if isinstance(n, ast.Call):
            f = n.func
            if   isinstance(f, ast.Name):      calls.append(f.id)
            elif isinstance(f, ast.Attribute): calls.append(f.attr)
        elif isinstance(n, ast.Import):
            imps += [a.name.split('.')[0] for a in n.names]
        elif isinstance(n, ast.ImportFrom):
            if n.module: imps.append(n.module.split('.')[0])
    keep = lambda s: s and s not in _PY_SKIP and not s.startswith('_') and len(s) > 2
    return _def_name(chunk), calls.filter(keep).unique(), imps.filter(keep).unique()

In [ ]:
from litesearch.data import dir2chunks
_chunks = dir2chunks('../litesearch', file_glob='*.py')
_gs = first(_chunks, lambda c: (c.get('metadata') or {}).get('name')=='get_store')
name, calls, imps = code_entities(_gs)
print(name, '->', list(calls))
assert name == 'get_store' and 'enable_fts' in calls

get_store -> ['create', 'update', 'detect_fts', 'enable_fts']


## Prose entities — spaCy noun_chunks, yake fallback

Sentence windows matter more than they look: PMI over page-sized chunks turns every entity pair into a clique and no amount of pruning recovers from it.

In [ ]:
#| export
_KEEP_ENTS = {'ORG','PRODUCT','PERSON','WORK_OF_ART','LAW','EVENT','GPE','NORP','FAC','SYMBOL'}

def spacy_pipe(model='en_core_web_sm',  # spaCy model name
               terms=None,              # exact-match terms for an EntityRuler (e.g. code symbols)
               label='SYMBOL',          # label applied to ruler matches
               download=True):          # fetch the model on first use if it isn't installed
    '''spaCy pipeline with an optional EntityRuler seeded from exact terms. None when spaCy is unavailable.
    The model is lazy-loaded and fetched once on first use — spaCy models aren't shippable as
    install-time deps (not on PyPI, and PyPI rejects URL deps), so we download on demand.'''
    try: import spacy
    except ImportError: return None
    try: nlp = spacy.load(model)
    except (OSError, IOError):
        if not download: return None
        try:
            from spacy.cli import download as _dl
            _dl(model); nlp = spacy.load(model)
        except Exception: return None
    if terms:
        pats = [{'label': label, 'pattern': t} for t in sorted(set(terms)) if t and len(t) > 2]
        if pats: nlp.add_pipe('entity_ruler', before='ner', config={'overwrite_ents': True}).add_patterns(pats)
    return nlp

def _yake_terms(text, topk=12):
    'Keyphrases via yake — the zero-model, zero-label fallback when spaCy is absent.'
    try: from yake import KeywordExtractor
    except ImportError: return L()
    try: return L(KeywordExtractor(n=3, top=topk).extract_keywords(text)).map(lambda kv: kv[0])
    except Exception: return L()

def prose_windows(text,                 # chunk text
                  nlp=None,             # spaCy pipeline from spacy_pipe(); None -> yake fallback
                  noun_chunks=True,     # use doc.noun_chunks (the main recall driver on technical prose)
                  keep=_KEEP_ENTS,      # spaCy ent labels kept as a `kind` facet
                  topk=12):             # yake fallback keyphrase count
    '''Entity surfaces grouped into co-occurrence windows — one per sentence when spaCy is available.
    Sentence windows are what make PMI meaningful: page-sized chunks turn every entity pair into a
    clique and no amount of pruning recovers from that.'''
    if nlp is None:
        # one yake pass for the chunk vocabulary, then place each phrase in the sentences it
        # occurs in — so the no-spaCy path gets real windows instead of one per chunk
        terms = _yake_terms(text, topk)
        if not terms: return L()
        wins = []
        for s in _sentences(text):
            sl = s.lower()
            hit = L([(t, 'keyphrase') for t in terms if t.lower() in sl])
            if hit: wins.append(hit)
        return L(wins)
    doc = nlp(text)
    wins = []
    for sent in doc.sents:
        out, seen = L(), set()
        def add(s, k):
            n = _norm(s)
            if n and n not in seen: seen.add(n); out.append((s, k))
        for e in sent.ents:
            if e.label_ in keep: add(e.text, e.label_.lower())
        if noun_chunks:
            for nc in sent.noun_chunks:
                if all(w.is_stop or w.is_punct or w.pos_ == 'PRON' for w in nc): continue
                add(nc.text, 'term')
        if out: wins.append(out)
    return L(wins)

def text_entities(text, nlp=None, **kw):
    'Entity surfaces for a prose chunk, flattened across windows. Returns L of (surface, kind).'
    seen, out = set(), L()
    for w in prose_windows(text, nlp, **kw):
        for s, k in w:
            n = _norm(s)
            if n and n not in seen: seen.add(n); out.append((s, k))
    return out

## Building the graph

In [ ]:
#| export
_CODE_TYPES = {'FunctionDef','AsyncFunctionDef','ClassDef'}

def _is_code(chunk):
    md = chunk.get('metadata') or {}
    if not isinstance(md, dict): return False
    return md.get('lang') == '.py' or md.get('type') in _CODE_TYPES

def _pmi_edges(wins,              # list of entity-id sets (one per co-occurrence window)
               min_n=2,           # min co-occurrence count
               min_npmi=0.15,     # min normalized PMI
               max_df=0.4,        # drop entities present in more than this fraction of windows
               max_degree=48,     # keep only the strongest edges per node
               rel='cooc'):
    'Normalized-PMI co-occurrence edges over windows. max_df is what kills hub terms like "section".'
    n = len(wins)
    if n < 2: return []
    cnt = {}
    for w in wins:
        for e in w: cnt[e] = cnt.get(e, 0) + 1
    hub = {e for e, c in cnt.items() if c / n > max_df}
    pair = {}
    for w in wins:
        es = sorted(e for e in w if e not in hub)
        for i, a in enumerate(es):
            for b in es[i+1:]: pair[(a, b)] = pair.get((a, b), 0) + 1
    out = []
    for (a, b), c in pair.items():
        if c < min_n: continue
        pa, pb, pab = cnt[a]/n, cnt[b]/n, c/n
        npmi = math.log(pab/(pa*pb)) / (-math.log(pab)) if 0 < pab < 1 else 0.0
        if npmi < min_npmi: continue
        out.append(dict(src=a, dst=b, rel=rel, weight=round(npmi, 5), n=c))
    if max_degree:
        deg, kept = {}, []
        for e in sorted(out, key=lambda r: -r['weight']):
            if deg.get(e['src'], 0) < max_degree and deg.get(e['dst'], 0) < max_degree:
                deg[e['src']] = deg.get(e['src'], 0)+1; deg[e['dst']] = deg.get(e['dst'], 0)+1
                kept.append(e)
        out = kept
    return out

def build_graph(db,                  # Database with a chunk store
                chunks,              # chunk dicts ({'content','metadata'}) as returned by dir2chunks/pkg2chunks
                store='store',       # chunk store name
                prefix=None,         # graph table prefix
                nlp=None,            # spaCy pipeline for prose (None -> yake fallback)
                emb_fn=None,         # embedder for entity names (required for resolve_entities)
                code=True,           # extract AST symbols from code chunks
                prose=True,          # extract surfaces from prose chunks
                cooc=True,           # also write PMI co-occurrence edges over sentence windows
                min_n=2,             # min co-occurrence count
                min_npmi=0.15,       # min normalized PMI
                max_df=0.4,          # drop entities present in >max_df of windows
                max_degree=48):      # max cooc edges kept per node
    'Extract entities + mentions + edges (exact for code, PMI co-occurrence for prose) from chunks.'
    g = db.get_graph(store, prefix)
    ents, mens, edges, wins = {}, {}, {}, []
    def ent(name, kind):
        'Register an entity by canonical name, returning its hash id.'
        n = _norm(name)
        if not n: return None
        i = _slug(n)
        e = ents.setdefault(i, dict(content=n, kind=kind, freq=0, canon=i))
        e['freq'] += 1
        return i
    def men(cid, eid, surface):
        if not (cid and eid): return
        m = mens.setdefault((cid, eid), dict(chunk_id=cid, entity_id=eid, surface=surface, n=0))
        m['n'] += 1
    def edge(s, d, rel, w=1.0):
        if not (s and d) or s == d: return
        e = edges.setdefault((s, d, rel), dict(src=s, dst=d, rel=rel, weight=0.0, n=0))
        e['weight'] += w; e['n'] += 1

    for c in L(chunks):
        txt = c.get('content')
        if not (txt and txt.strip()): continue
        cid = _slug(txt)
        if code and _is_code(c):
            dname, calls, imps = code_entities(c)
            did = ent(dname, 'symbol') if dname else None
            if did: men(cid, did, dname)
            w = {did} if did else set()
            for nm in calls:
                i = ent(nm, 'symbol'); men(cid, i, nm); edge(did, i, 'calls'); w.add(i)
            for nm in imps:
                i = ent(nm, 'module'); men(cid, i, nm); edge(did, i, 'imports'); w.add(i)
            if len(w) > 1: wins.append(w - {None})
        elif prose:
            for win in prose_windows(txt, nlp):
                w = set()
                for surf, kind in win:
                    i = ent(surf, kind)
                    if i: men(cid, i, surf); w.add(i)
                if len(w) > 1: wins.append(w)

    rows = list(ents.values())
    if cooc and wins:
        for e in _pmi_edges(wins, min_n, min_npmi, max_df, max_degree):
            edges[(e['src'], e['dst'], e['rel'])] = e
    if rows:
        if emb_fn: process_content(g.entities, rows, embed=True, emb_fn=emb_fn)
        else:      g.entities.insert_all(rows, upsert=True, hash_id='id', hash_id_columns=['content'])
    if mens:  g.mentions.insert_all(list(mens.values()), upsert=True, pk=('chunk_id','entity_id'))
    if edges: g.edges.insert_all(list(edges.values()), upsert=True, pk=('src','dst','rel'))
    if emb_fn and rows: g.entities.rebuild_index()
    return dict(entities=len(rows), mentions=len(mens), edges=len(edges), windows=len(wins))

## Entity resolution

In [ ]:
#| export
_EXACT_KINDS = ('symbol', 'module', 'topic')   # names that are already canonical — never merge these

def _uf_find(par, x):
    while par[x] != x: par[x] = par[par[x]]; x = par[x]
    return x

def _uf_union(par, rank, a, b):
    ra, rb = _uf_find(par, a), _uf_find(par, b)
    if ra == rb: return False
    if rank[ra] < rank[rb]: ra, rb = rb, ra
    par[rb] = ra
    return True

def _lexical_pairs(name, max_group=60):
    '''Candidate merge pairs by shared-token blocking — catches containment variants
    ("isolated polonium" / "polonium") that an ANN pass on short strings misses.
    Groups larger than max_group are skipped: common tokens would make this quadratic.'''
    inv = {}
    for i, s in name.items():
        for t in _toks(s):
            if len(t) > 2: inv.setdefault(t, []).append(i)
    seen = set()
    for ids in inv.values():
        if not (2 <= len(ids) <= max_group): continue
        for a in range(len(ids)):
            for b in range(a+1, len(ids)):
                p = (ids[a], ids[b]) if ids[a] < ids[b] else (ids[b], ids[a])
                if p not in seen: seen.add(p); yield p

def resolve_entities(db,                # Database
                     store='store',     # chunk store the graph belongs to
                     prefix=None,       # graph table prefix
                     thresh=0.18,       # max ANN distance for a merge candidate
                     lex=0.34,          # min token Jaccard for the lexical guard
                     k=8,               # ANN neighbours considered per entity
                     lexical=True,      # also block on shared tokens (works without embeddings)
                     max_group=60,      # skip token groups bigger than this in the lexical pass
                     skip_kinds=_EXACT_KINDS,  # kinds whose names are already canonical
                     dtype=np.float16):
    '''Merge near-duplicate entities: ANN + shared-token candidates, both gated by the lexical guard.
    Kinds in `skip_kinds` are left alone — AST symbols are exact identifiers, and resolving them
    collapses `search`/`fts_search`/`vec_search` into one node.'''
    g = db.get_graph(store, prefix)
    allr = L(g.entities(select='id, content, freq, embedding, kind'))
    rows = allr.filter(lambda r: r['kind'] not in set(skip_kinds or ()))
    if len(rows) < 2:
        return dict(merged=0, by_ann=0, by_lexical=0, edges=len(list(g.edges())),
                    entities=len(allr), resolvable=len(rows), canonical=len(allr))
    par  = {r['id']: r['id'] for r in rows}
    rank = {r['id']: (r['freq'] or 0) for r in rows}
    name = {r['id']: r['content'] for r in rows}
    ann_m = lex_m = 0
    for r in rows.filter(lambda r: r['embedding']):
        for h in g.entities.ann_search(r['embedding'], columns=['id','content'], limit=k, dtype=dtype):
            oid = h.get('id')
            if not oid or oid == r['id'] or oid not in par: continue
            if (h.get('_dist') or 1.0) > thresh: continue
            if not _lex_ok(r['content'], h['content'], lex): continue
            if _uf_union(par, rank, r['id'], oid): ann_m += 1
    if lexical:
        for a, b in _lexical_pairs(name, max_group):
            if _lex_ok(name[a], name[b], lex) and _uf_union(par, rank, a, b): lex_m += 1
    upd = [(_uf_find(par, i), i) for i in par]
    db.conn.cursor().executemany(f'update {g.entities.name} set canon=? where id=?', upd)
    canon = {i: c for c, i in upd}
    n_edges = _collapse_edges(db, g, canon)
    skipped = len(allr) - len(rows)
    return dict(merged=ann_m+lex_m, by_ann=ann_m, by_lexical=lex_m, edges=n_edges,
                entities=len(allr), resolvable=len(rows),
                canonical=len({c for c, _ in upd}) + skipped)

def _collapse_edges(db, g, canon):
    '''Rewrite edge endpoints onto canonical ids. Traversal reads src/dst straight from the table,
    so leaving raw ids here would silently strand every merged node.'''
    rows = list(g.edges())
    if not rows: return 0
    agg = {}
    for r in rows:
        s, d = canon.get(r['src'], r['src']), canon.get(r['dst'], r['dst'])
        if s == d: continue
        if s > d and r['rel'] == 'cooc': s, d = d, s     # cooc is symmetric; keep one direction
        k = (s, d, r['rel'])
        a = agg.setdefault(k, dict(src=s, dst=d, rel=r['rel'], weight=0.0, n=0))
        a['weight'] = max(a['weight'], r['weight'] or 0.0); a['n'] += (r['n'] or 0)
    g.edges.delete_where()
    g.edges.insert_all(list(agg.values()), upsert=True, pk=('src','dst','rel'))
    return len(agg)

`cooccur_edges` recomputes edges from stored mentions using the whole chunk as the window — use it after `resolve_entities`, or for corpora indexed without a spaCy pass.

In [ ]:
#| export
def cooccur_edges(db,                 # Database
                  store='store',      # chunk store
                  prefix=None,        # graph table prefix
                  min_n=2,            # min co-occurrence count
                  min_npmi=0.15,      # min normalized PMI (prunes stopword-ish hub nodes)
                  max_df=0.4,         # drop entities present in >max_df of chunks
                  max_degree=48,      # keep only the strongest edges per node
                  rel='cooc',
                  use_canon=True):    # collapse to canonical ids from resolve_entities
    '''Rebuild co-occurrence edges from the stored mentions, using the chunk as the window.
    build_graph already does this over sentence windows; use this to recompute after
    resolve_entities, or for corpora indexed without a spaCy pass.'''
    g = db.get_graph(store, prefix)
    canon = {}
    if use_canon:
        canon = {r['id']: (r['canon'] or r['id']) for r in g.entities(select='id, canon')}
    cid_ents = {}
    for m in g.mentions(select='chunk_id, entity_id'):
        e = canon.get(m['entity_id'], m['entity_id'])
        cid_ents.setdefault(m['chunk_id'], set()).add(e)
    out = _pmi_edges(list(cid_ents.values()), min_n, min_npmi, max_df, max_degree, rel)
    if out: g.edges.insert_all(out, upsert=True, pk=('src','dst','rel'))
    return len(out)

## Topics from the ANN index

`usearch` clusters straight off the HNSW graph, but it walks index levels and raises
`Index too small to cluster!` on small corpora — so we fall back to harvesting the kNN graph and
greedily seeding clusters, which works at any size.

Labels are c-TF-IDF: term frequency inside a cluster weighted by inverse document frequency **across
the clusters**. Plain frequency names every cluster after the same handful of words the corpus is
made of; the cross-cluster IDF is what makes the names differ from each other, which is their only
job. (Approach taken from `lego/atlas/cluster.py`.)

In [ ]:
#| export
# connector words — cluster names built from these describe the corpus, not the cluster
_STOP = set('''the a an and or of to in is are was were be been being for on at by with from this that these those
it its as not but if then than so such can will would could should may might do does did have has had he she they
we you i him her them our your their about into over under after before between out up down off only own same too
very just also each other more most some any no nor own s t don now here there when where why how all both'''.split())
_LWORD = re.compile(r'[A-Za-z_][A-Za-z0-9_]{2,}')   # keeps identifiers whole

def ctfidf_labels(texts,        # one text per member, aligned with `lab`
                  lab,          # cluster index per member
                  k,            # number of clusters
                  top_n=4,      # terms per label
                  stop=None,    # stopword set (defaults to _STOP)
                  sep=', '):
    '''Name each cluster by terms common inside it and rare across the other clusters.

    Plain term frequency names every cluster after the same few words the corpus is made of;
    weighting by IDF across the *clusters* is what makes the names differ from each other,
    which is the only job they have. (Approach taken from lego/atlas/cluster.py.)'''
    st = _STOP if stop is None else stop
    tf = [{} for _ in range(k)]
    for t, j in zip(texts, lab):
        for w in set(_LWORD.findall((t or '').lower())):
            if w in st or len(w) < 3: continue
            tf[j][w] = tf[j].get(w, 0) + 1
    df = {}
    for d in tf:
        for w in d: df[w] = df.get(w, 0) + 1
    out = []
    for j, d in enumerate(tf):
        n = max(1, sum(1 for x in lab if x == j))
        sc = {w: (c/n) * math.log(k/df[w]) for w, c in d.items() if df[w] < k}
        top = sorted(sc.items(), key=lambda kv: -kv[1])[:top_n]
        out.append(sep.join(w for w, _ in top))
    return out

def _usearch_clusters(idx, min_count, max_count):
    '''(centroid, members) straight off the HNSW graph. usearch walks the index levels, so it needs
    a tall enough graph and raises "Index too small to cluster!" on small corpora — caller falls back.'''
    kw = {k: v for k, v in dict(min_count=min_count, max_count=max_count).items() if v}
    cl = idx.cluster(**kw)
    keys, _ = cl.centroids_popularity
    out = L()
    for ck in np.atleast_1d(keys).tolist():
        try: out.append((int(ck), [int(x) for x in np.atleast_1d(cl.members_of(ck)).tolist()]))
        except Exception: continue
    return out

def _knn_clusters(idx, keys, k=8, max_size=None, min_sim=None, dtype=np.float16):
    '''(seed, members) from a greedy pass over the kNN graph harvested from HNSW. Works at any size.
    Seeds at the densest unassigned node and claims its unassigned neighbours, so clusters stay
    bounded — plain label propagation collapses a dense kNN graph into one giant component.'''
    keys = list(keys)
    if len(keys) < 4: return L()
    vecs = np.stack([np.asarray(idx[kk], dtype=dtype).reshape(-1) for kk in keys])
    res = idx.search(vecs, count=min(k+1, len(keys)))
    nbr, sims = {}, []
    for i, kk in enumerate(keys):
        ks = np.atleast_1d(res.keys[i]).tolist()
        ds = np.atleast_1d(res.distances[i]).tolist()
        n = [(int(a), 1.0-float(b)) for a, b in zip(ks, ds) if int(a) != kk]
        nbr[kk] = n
        sims += [s for _, s in n]
    if min_sim is None: min_sim = float(np.median(sims)) if sims else 0.0
    cap = max_size or max(k, 4)
    dens = {kk: sum(s for _, s in v if s >= min_sim) for kk, v in nbr.items()}
    out, seen = L(), set()
    for kk in sorted(keys, key=lambda x: -dens.get(x, 0.0)):
        if kk in seen: continue
        grp = [kk]; seen.add(kk)
        for nk, s in sorted(nbr.get(kk, []), key=lambda t: -t[1]):
            if len(grp) >= cap: break
            if nk not in seen and s >= min_sim: grp.append(nk); seen.add(nk)
        out.append((int(kk), [int(x) for x in grp]))
    return out

def _cluster_groups(idx,               # usearch Index
                    keys=None,         # keys to cluster (defaults to everything in the index)
                    min_count=None,    # usearch: smallest cluster to emit
                    max_count=None,    # usearch: largest cluster to emit
                    k=8,               # neighbours per node in the kNN fallback
                    dtype=np.float16):
    '''`([(centroid, members)], method)` for an index.

    usearch raises rather than degrading when the HNSW graph has too few levels to cut, so the
    greedy kNN pass is not a nicety: without it clustering is simply unavailable on a fresh or
    small store, which is exactly when someone is most likely to try it.'''
    try: return _usearch_clusters(idx, min_count, max_count), 'usearch'
    except Exception:
        ks = np.atleast_1d(idx.keys).tolist() if keys is None else list(keys)
        return _knn_clusters(idx, ks, k, dtype=dtype), 'knn'

def topic_nodes(db,                # Database
                store='store',     # chunk store (must be ANN-registered)
                prefix=None,       # graph table prefix
                min_count=None,    # usearch cluster min size
                max_count=None,    # usearch cluster max size
                k=8,               # neighbours per node in the fallback kNN graph
                min_size=2,        # smallest cluster kept as a topic
                label_k=4,         # terms per topic label
                max_label_chars=4000,
                dtype=np.float16):
    'Cluster the store index into topic nodes labelled by c-TF-IDF. Returns {topics, method}.'
    g = db.get_graph(store, prefix)
    m = db._ann_meta(store)
    if not (m and m['ndim']): return dict(topics=0, method=None)
    idx = db.get_index(store)
    if idx.size < 4: return dict(topics=0, method=None)
    dt = _np_dtype.get(m['dtype'], dtype)
    rid2cid = {r['rowid']: r['id'] for r in db.t[store](select=f'{_rid()}, id')}
    groups, method = _cluster_groups(idx, rid2cid.keys(), min_count, max_count, k, dt)
    kept = [[rid2cid[r] for r in mem if r in rid2cid] for _, mem in groups]
    kept = [c for c in kept if len(c) >= min_size]
    if not kept: return dict(topics=0, method=method)
    # labels are scored across all clusters at once, so gather members first
    txt = {r['id']: r['content'] for r in db.t[store](select='id, content',
                                                      where=_in('id', [c for g_ in kept for c in g_[:24]]))}
    texts, lab = [], []
    for j, cids in enumerate(kept):
        for c in cids[:24]: texts.append((txt.get(c) or '')[:max_label_chars]); lab.append(j)
    names = ctfidf_labels(texts, lab, len(kept), label_k)
    ents, mens = [], []
    for j, cids in enumerate(kept):
        # deliberately not _norm(): topic labels are multi-phrase and would trip its 5-token cap
        name = f'topic: {names[j]}'[:60] if names[j] else f'topic-{j}'
        eid = _slug(name)
        ents.append(dict(content=name, kind='topic', freq=len(cids), canon=eid))
        mens += [dict(chunk_id=c, entity_id=eid, surface=name, n=1) for c in cids]
    if ents: g.entities.insert_all(ents, upsert=True, hash_id='id', hash_id_columns=['content'])
    if mens: g.mentions.insert_all(mens, upsert=True, pk=('chunk_id','entity_id'))
    return dict(topics=len(ents), method=method)

## Clusters and peers — the corpus by shape

`topic_nodes` writes clusters into the graph as topic entities. The same machinery answers two
questions directly, without a graph: **`store.clusters()`** maps the whole corpus (one labelled
group per region of embedding space) and **`store.peers(rowid)`** returns the group one row
belongs to.

The distinction between `peers` and `ann_neighbors` is worth keeping straight. k-NN answers *"the
15 things closest to this"*, and always returns 15 whether or not they are related. A cluster
answers *"the family this belongs to"* — which is the question behind "where else did we already
do this?". `peers` falls back to `ann_neighbors` when the index cannot be clustered, so the
feature degrades instead of disappearing, and says so in `note`.

Both return `note`, and that is deliberate: a clustering that silently returns an empty list is
indistinguishable from a broken index, and the caller in front of a user has to show *something*.

In [ ]:
#| export
@patch
def _cluster_cached(self:Table, min_count, max_count, k, dtype):
    'Cluster once per (store, params, index size). `peers` would otherwise recluster on every call.'
    idx = self.db.get_index(self.name)
    ck = (self.name, min_count, max_count, k, idx.size)
    cache = getattr(self.db, '_cluster_cache', None)
    if cache is None: cache = self.db._cluster_cache = {}
    if ck not in cache:
        groups, method = _cluster_groups(idx, None, min_count, max_count, k, dtype)
        assign = {m: g for _, g in groups for m in g}
        cache[ck] = (groups, method, assign)
    return cache[ck]

@patch
def _member_rows(self:Table, keys, columns=None):
    'Store rows for a list of usearch keys, keyed by rowid. `content` is always fetched (labels need it).'
    if not keys: return {}
    cols = list(dict.fromkeys([c for c in (columns or []) if c != 'rowid'] + ['content']))
    sel, out = ','.join(cols + [_rid()]), {}
    for i in range(0, len(keys), 400):
        for r in self.db.q(f'select {sel} from {self.name} where {_in("rowid", keys[i:i+400])}'): out[r['rowid']] = r
    return out

@patch
def clusters(self:Table,              # ANN-registered store
             min_count:int=None,      # usearch: smallest cluster to emit
             max_count:int=None,      # usearch: largest cluster to emit
             k:int=8,                 # neighbours per node in the kNN fallback
             min_size:int=2,          # drop groups smaller than this
             label_k:int=4,           # terms per c-TF-IDF label
             members:int=24,          # member rows fetched per cluster
             columns:list=None,       # store columns to return per member row
             max_label_chars:int=4000,# per-member text budget for labelling
             dtype=np.float16):
    '''The corpus grouped by embedding shape, each group named by c-TF-IDF.

    Returns `AttrDict(clusters, method, note)`; each cluster is
    `AttrDict(centroid, size, label, member_keys, members)`. `method` is `usearch` or the `knn` fallback.'''
    m = self.db._ann_meta(self.name)
    if not m: return AttrDict(clusters=L(), method=None, note=f'{self.name!r} is not an ANN store')
    if not m['ndim']: return AttrDict(clusters=L(), method=None, note=f'{self.name!r} has no vectors yet')
    idx = self.db.get_index(self.name)
    if idx.size < 4: return AttrDict(clusters=L(), method=None, note=f'index holds {idx.size} vectors; too few to cluster')
    dt = _np_dtype.get(m['dtype'], dtype)
    groups, method, _ = self._cluster_cached(min_count, max_count, k, dt)
    kept = [(c, g) for c, g in groups if len(g) >= min_size]
    if not kept: return AttrDict(clusters=L(), method=method, note=f'no group reached min_size={min_size}')
    rows = self._member_rows([r for _, g in kept for r in g[:members]], columns)
    texts, lab = [], []
    for j, (_, g) in enumerate(kept):
        for r in g[:members]: texts.append(((rows.get(r) or {}).get('content') or '')[:max_label_chars]); lab.append(j)
    names = ctfidf_labels(texts, lab, len(kept), label_k)
    out = L(AttrDict(centroid=c, size=len(g), label=names[j] or f'group-{j}', member_keys=g,
                     members=L(rows[r] for r in g[:members] if r in rows))
            for j, (c, g) in enumerate(kept))
    return AttrDict(clusters=out.sorted(key=lambda c: -c.size), method=method,
                    note=f'{len(out)} clusters over {idx.size} vectors ({method})')

@patch
def peers(self:Table,            # ANN-registered store
          key:int,               # usearch key (rowid) whose group you want
          limit:int=25,          # members to return
          columns:list=None,     # store columns to return per member row
          min_count:int=None,    # usearch: smallest cluster to emit
          max_count:int=None,    # usearch: largest cluster to emit
          k:int=8,               # neighbours per node in the kNN fallback
          dtype=np.float16):
    '''The group `key` belongs to — its family, not a ranked list of what is nearest to it.

    Degrades to `ann_neighbors` (and says so in `note`) whenever the index cannot be clustered or
    the row landed in a group of one. Returns `AttrDict(hits, method, note)`.'''
    nbr = lambda note: AttrDict(hits=L(self.ann_neighbors(key, limit, columns, dtype=dtype)), method='ann', note=note)
    m = self.db._ann_meta(self.name)
    if not m: return AttrDict(hits=L(), method=None, note=f'{self.name!r} is not an ANN store')
    if not m['ndim']: return AttrDict(hits=L(), method=None, note=f'{self.name!r} has no vectors yet')
    idx = self.db.get_index(self.name)
    if not idx.size or not idx.contains(key): return AttrDict(hits=L(), method=None, note=f'key {key} is not indexed')
    dt = _np_dtype.get(m['dtype'], dtype)
    if idx.size < 4: return nbr(f'index holds {idx.size} vectors; showing nearest neighbours')
    _, method, assign = self._cluster_cached(min_count, max_count, k, dt)
    grp = assign.get(key)
    if not grp: return nbr('row is in no cluster; showing nearest neighbours')
    mem = [x for x in grp if x != key][:limit]
    if not mem: return nbr('cluster has one member; showing nearest neighbours')
    rows, order = self._member_rows(mem, columns), {r: i for i, r in enumerate(mem)}
    return AttrDict(hits=L(rows[r] for r in sorted(rows, key=lambda r: order.get(r, 1<<30)) if r in rows),
                    method=method, note=f'cluster of {len(grp)} ({method})')

In [ ]:
# clusters(): two well-separated blobs -> two groups, each labelled by its own vocabulary
from litesearch.core import database
_cdb = database()
_cst = _cdb.get_store('cl', ann=True, ndim=8, metric='cosine')
_rs  = np.random.RandomState(0)
_a   = np.concatenate([np.ones((30,4)), np.zeros((30,4))], 1) + _rs.randn(30,8)*0.05
_b   = np.concatenate([np.zeros((30,4)), np.ones((30,4))], 1) + _rs.randn(30,8)*0.05
_txt = [f'kernel gradient tensor sample {i}' for i in range(30)] + [f'invoice ledger payment row {i}' for i in range(30)]
_cst.insert_all([{'content':t,'embedding':v.astype(np.float16).tobytes()}
                 for t,v in zip(_txt, np.concatenate([_a,_b]))])
assert _cst.rebuild_index() == 60
_cl = _cst.clusters(min_size=3)
print(_cl.note, '|', [(c.size, c.label) for c in _cl.clusters][:4])
assert len(_cl.clusters) >= 2, _cl.note
assert _cl.method in ('usearch','knn')
_labels = ' '.join(c.label for c in _cl.clusters)
assert 'tensor' in _labels or 'kernel' in _labels, _labels     # c-TF-IDF names a group after what only it says
assert all(r['rowid'] in c.member_keys for c in _cl.clusters for r in c.members)

# peers(): every member of a group has the rest of that group as its family
_c0 = first(_cl.clusters, lambda c: 'kernel' in c.label or 'tensor' in c.label)
_pr = _cst.peers(_c0.centroid, limit=5, columns=['content'])
print(_pr.note, '|', [h['content'][:24] for h in _pr.hits])
assert _pr.hits and _pr.method == _cl.method, _pr.note
assert all('kernel' in h['content'] for h in _pr.hits), _pr.hits
# a row the clustering left on its own still gets an answer -- with the fallback named in `note`
_solo = first(range(1, 61), lambda r: not _cst.peers(r).method == _cl.method)
if _solo: assert 'nearest neighbours' in _cst.peers(_solo).note

# degradation is reported, never silent
_tiny = database().get_store('tiny', ann=True, ndim=4, metric='cosine')
_tiny.insert_all([{'content':'x','embedding':np.zeros(4,dtype=np.float16).tobytes()}])
_tiny.rebuild_index()
assert _tiny.clusters().clusters == [] and 'too few' in _tiny.clusters().note
assert 'nearest neighbours' in _tiny.peers(1).note
assert database().get_store('plain').clusters().note.endswith('is not an ANN store')
assert database().get_store('empty', ann=True).clusters().note.endswith('has no vectors yet')

## Traversal, PPR and fusion

The graph becomes a third RRF leg beside FTS and vectors.

In [ ]:
#| export
def _adjacency(g, nodes, hops=2, max_nodes=4000):
    'BFS the edge table out to `hops` from `nodes`; returns an undirected dict-of-dict adjacency.'
    adj, seen, frontier = {}, set(nodes), set(nodes)
    for _ in range(max(hops, 0)):
        if not frontier or len(seen) >= max_nodes: break
        fl = list(frontier)
        rows = []
        for i in range(0, len(fl), 400):
            b = fl[i:i+400]
            rows += g.edges(where=f"{_in('src', b)} OR {_in('dst', b)}")
        nxt = set()
        for r in rows:
            s, d, w = r['src'], r['dst'], (r['weight'] or 1.0)
            adj.setdefault(s, {})[d] = max(adj.setdefault(s, {}).get(d, 0.0), w)
            adj.setdefault(d, {})[s] = max(adj.setdefault(d, {}).get(s, 0.0), w)
            for x in (s, d):
                if x not in seen: nxt.add(x)
        seen |= nxt
        frontier = nxt
    return adj

def _ppr(adj, seeds, damping=0.85, iters=12):
    'Personalized PageRank over a dict-of-dict adjacency.'
    if not seeds: return {}
    tot = sum(seeds.values()) or 1.0
    p0 = {k: v/tot for k, v in seeds.items()}
    r = dict(p0)
    for _ in range(iters):
        nxt = {}
        for u, mass in r.items():
            nb = adj.get(u)
            if not nb: continue
            s = sum(nb.values()) or 1.0
            for v, w in nb.items(): nxt[v] = nxt.get(v, 0.0) + damping*mass*w/s
        for k, v in p0.items(): nxt[k] = nxt.get(k, 0.0) + (1-damping)*v
        r = nxt
    return r

In [ ]:
#| export
def rrf_all(lists,              # list of ranked result lists
            k=60,               # RRF k
            limit=50,           # max results
            id_key='rowid',     # join key
            weights=None):      # per-list weights
    'Reciprocal Rank Fusion over any number of ranked lists.'
    ws = weights or [1.0]*len(lists)
    scores = {}
    for lst, w in zip(lists, ws):
        for rank, row in enumerate(lst or []):
            rid = row.get(id_key, id(row))
            if rid in scores: scores[rid]['_rrf_score'] += w/(k + rank)
            else: scores[rid] = merge(row, {'_rrf_score': w/(k + rank)})
    return sorted(scores.values(), key=lambda x: x['_rrf_score'], reverse=True)[:limit]

In [ ]:
#| export
@patch
def graph_search(self:Database,
                 q:str,                 # query string
                 emb:bytes,             # query embedding
                 columns:list=None,     # columns to return
                 limit:int=20,          # max results
                 table_name='store',    # chunk store
                 prefix=None,           # graph table prefix
                 seed_n:int=12,         # hybrid hits used to seed the graph walk
                 hops:int=2,            # edge-table BFS depth
                 damping:float=0.85,    # PPR damping
                 iters:int=10,          # PPR iterations
                 graph_w:float=0.5,     # weight of the graph leg (low by default — see docstring)
                 rrf_k:int=60,
                 use_canon=True,
                 **kw):                 # forwarded to Database.search
    '''Hybrid search plus a graph leg: PPR over the entity graph seeded by the top hybrid hits.

    The graph leg is weighted low on purpose. It pays when the answer shares no vocabulary with
    the query and is reachable only along an entity path — common in prose, rare in code, where
    call edges link different levels of abstraction rather than substitutable answers. On code
    corpora prefer `graph_w=0` and use the graph for context assembly (callers/callees of a hit)
    rather than for ranking.'''
    g = self.get_graph(table_name, prefix)
    cols = list(columns or [])
    if 'rowid' not in cols: cols = ['rowid'] + cols
    if 'id' not in cols: cols = cols + ['id']
    base = self.search(q, emb, columns=cols, limit=max(seed_n*3, limit), table_name=table_name,
                       rrf=False, **kw)
    if not base: return []
    fts, vec = base['fts'], base['vec']
    seed_rows = rrf_all([fts, vec], rrf_k, seed_n)
    seed_cids = [r['id'] for r in seed_rows if r.get('id')]
    if not seed_cids: return rrf_all([fts, vec], rrf_k, limit)
    canon = {}
    if use_canon: canon = {r['id']: (r['canon'] or r['id']) for r in g.entities(select='id, canon')}
    seeds = {}
    for i in range(0, len(seed_cids), 400):
        for m in g.mentions(select='chunk_id, entity_id', where=_in('chunk_id', seed_cids[i:i+400])):
            e = canon.get(m['entity_id'], m['entity_id'])
            seeds[e] = seeds.get(e, 0.0) + 1.0
    if not seeds: return rrf_all([fts, vec], rrf_k, limit)
    adj = _adjacency(g, set(seeds), hops)
    mass = _ppr(adj, seeds, damping, iters)
    if not mass: return rrf_all([fts, vec], rrf_k, limit)
    top = sorted(mass.items(), key=lambda kv: -kv[1])[:200]
    eids = [e for e, _ in top if _ > 0]
    if not eids: return rrf_all([fts, vec], rrf_k, limit)
    inv = {}
    for i in range(0, len(eids), 400):
        for m in g.mentions(select='chunk_id, entity_id', where=_in('entity_id', eids[i:i+400])):
            e = canon.get(m['entity_id'], m['entity_id'])
            inv[m['chunk_id']] = inv.get(m['chunk_id'], 0.0) + mass.get(e, 0.0)
    if not inv: return rrf_all([fts, vec], rrf_k, limit)
    ranked = sorted(inv.items(), key=lambda kv: -kv[1])[:limit*3]
    cids = [c for c, _ in ranked]
    sel = ','.join([_rid() if c == 'rowid' else c for c in cols])
    rows = {r['id']: r for r in self.t[table_name](select=sel, where=_in('id', cids))}
    graph_leg = [rows[c] for c, _ in ranked if c in rows]
    return rrf_all([fts, vec, graph_leg], rrf_k, limit, weights=[1.0, 1.0, graph_w])

In [ ]:
#| export
def graph_stats(db, store='store', prefix=None):
    'Row counts and top-degree nodes for a built graph.'
    g = db.get_graph(store, prefix)
    ne = first(db.q(f'select count(*) c from {g.entities.name}'))['c']
    nm = first(db.q(f'select count(*) c from {g.mentions.name}'))['c']
    ng = first(db.q(f'select count(*) c from {g.edges.name}'))['c']
    nc = first(db.q(f'select count(distinct canon) c from {g.entities.name}'))['c']
    top = db.q(f'''select e.content, e.kind, count(*) d from {g.edges.name} g
                   join {g.entities.name} e on e.id=g.src group by g.src order by d desc limit 10''')
    return dict(entities=ne, canonical=nc, mentions=nm, edges=ng, top_degree=top)

## End to end: code

The litesearch package indexed against itself.

In [ ]:
import os, tempfile
from litesearch import database, doc_encoder, static_code_embedder
from fastcore.all import Path

_tmp = tempfile.mkdtemp()
emb = lambda txts, **kw: hash_embed(txts)
db = database(f'{_tmp}/code.db')
store = db.get_store(hash=True, ann=True)
rows = [dict(content=c['content'], metadata=str(c['metadata'])) for c in _chunks if c['content'].strip()]
store.insert_all([dict(r, embedding=e.tobytes()) for r,e in zip(rows, emb([r['content'] for r in rows]))],
                 upsert=True, hash_id='id', hash_id_columns=['content'])
store.rebuild_index()

print('build   :', build_graph(db, _chunks, prose=False, emb_fn=emb))
print('resolve :', resolve_entities(db))
print('topics  :', topic_nodes(db))
st = graph_stats(db)
print('stats   :', {k:v for k,v in st.items() if k!='top_degree'})
for t in st['top_degree'][:6]: print(f"   {t['d']:3d}  {t['content'][:44]}")
assert st['entities'] > 100 and st['edges'] > 50

build   : {'entities': 307, 'mentions': 585, 'edges': 769, 'windows': 99}
resolve : {'merged': 0, 'by_ann': 0, 'by_lexical': 0, 'edges': 769, 'entities': 307, 'resolvable': 0, 'canonical': 307}
topics  : {'topics': 19, 'method': 'knn'}
stats   : {'entities': 326, 'canonical': 326, 'mentions': 635, 'edges': 769}
    36  fastencode
    27  fastencodeimage
    23  download_model
    20  zeros
    19  path
    17  cpu_count


Hub symbols come out as the real ones — `FastEncode`, `download_model`, `pyparse` — because the edges are AST calls, not guesses.

## End to end: PDF

Sentence windows vs page-sized chunks, measured.

In [ ]:
from litesearch.data import file_parse
import litesearch.graph as _G
emb = doc_encoder(static_code_embedder())
_pdf = Path('pdfs/attention_is_all_you_need.pdf')
pchunks = [c for c in file_parse(_pdf) if c['content'].strip()]
nlp = spacy_pipe()
print('spacy:', 'ok' if nlp else 'not installed -> yake fallback', '| chunks:', len(pchunks))

pdb = database(f'{_tmp}/pdf.db')
pstore = pdb.get_store(hash=True, ann=True)
prows = [dict(content=c['content'], metadata=str(c['metadata'])) for c in pchunks]
pstore.insert_all([dict(r, embedding=e.tobytes()) for r,e in zip(prows, emb([r['content'] for r in prows]))],
                  upsert=True, hash_id='id', hash_id_columns=['content'])
pstore.rebuild_index()

print('build   :', build_graph(pdb, pchunks, code=False, nlp=nlp, emb_fn=emb))
print('resolve :', resolve_entities(pdb))
pst = graph_stats(pdb)
print('stats   :', {k:v for k,v in pst.items() if k!='top_degree'})
print('top entities:', [t['content'][:28] for t in pst['top_degree'][:8]])

# contrast: the same PMI over page-sized windows
canon = {r['id']:(r['canon'] or r['id']) for r in pdb.t.entities(select='id, canon')}
per_chunk = {}
for m in pdb.t.mentions(select='chunk_id, entity_id'):
    per_chunk.setdefault(m['chunk_id'], set()).add(canon.get(m['entity_id'], m['entity_id']))
print(f"edges, sentence windows : {pst['edges']}")
print(f"edges, page windows     : {len(_G._pmi_edges(list(per_chunk.values())))}   <- clique blowup")
assert pst['entities'] > 50

Dictionary used where Stream expected, treating as empty stream
Dictionary used where Stream expected, treating as empty stream
Dictionary used where Stream expected, treating as empty stream
Dictionary used where Stream expected, treating as empty stream


spacy: ok | chunks: 18
build   : {'entities': 1108, 'mentions': 1366, 'edges': 115, 'windows': 339}
resolve : {'merged': 389, 'by_ann': 131, 'by_lexical': 258, 'edges': 75, 'entities': 1108, 'resolvable': 1108, 'canonical': 719}
stats   : {'entities': 1108, 'canonical': 719, 'mentions': 1366, 'edges': 75}
top entities: ['wmt', 'decoder', 'bleu', 'french', 'transformer', 'queries', 'layer', 'wmt 2014 english-to-german t']
edges, sentence windows : 75
edges, page windows     : 846   <- clique blowup


## What the graph leg actually adds

The query names Marie Curie. Doc **B** never mentions her — it is reachable only by walking
`Marie Curie -> polonium -> B`. The structural assertion below holds regardless of embedding
quality; the ranking print is indicative only, since these tests run on the hashing embedder.

In [ ]:
DOCS = {
 'A': 'Marie Curie isolated polonium in 1898. Marie Curie worked in Paris.',
 'B': 'Polonium is intensely radioactive. Polonium decays by alpha emission.',
 'C': 'Alpha emission was studied by Ernest Rutherford at Manchester.',
 'D': 'The rainfall in Bergen is heavy. Bergen has a maritime climate.',
 'E': 'Bergen is a port city. Ships dock at Bergen harbour daily.',
 'F': 'Cricket is played with a bat. The bat is made of willow wood.',
 'G': 'Willow wood grows near rivers. Rivers flood in the spring season.',
 'H': 'Spring season brings warm weather. Warm weather melts the snow.'}
bchunks = [dict(content=v, metadata=dict(doc=k, lang='.txt')) for k,v in DOCS.items()]

bdb = database(f'{_tmp}/bridge.db')
bstore = bdb.get_store(hash=True, ann=True)
bstore.insert_all([dict(content=c['content'], metadata=str(c['metadata']), embedding=e.tobytes())
                   for c,e in zip(bchunks, emb([c['content'] for c in bchunks]))],
                  upsert=True, hash_id='id', hash_id_columns=['content'])
bstore.rebuild_index()
build_graph(bdb, bchunks, code=False, nlp=nlp, emb_fn=emb, min_n=1, min_npmi=-1.0, max_df=1.0)
resolve_entities(bdb)

bcanon = {r['id']:(r['canon'] or r['id']) for r in bdb.t.entities(select='id, canon')}
bname  = {r['id']:r['content'] for r in bdb.t.entities(select='id, content')}
doc_of = {r['id']: {v:k for k,v in DOCS.items()}[r['content']] for r in bdb.t.store(select='id, content')}
d2e = {}
for m in bdb.t.mentions(select='chunk_id, entity_id'):
    d2e.setdefault(doc_of[m['chunk_id']], set()).add(bcanon.get(m['entity_id'], m['entity_id']))

shared = d2e['A'] & d2e['B']
print('A entities:', sorted(bname[e] for e in d2e['A']))
print('B entities:', sorted(bname[e] for e in d2e['B']))
print('shared    :', sorted(bname[e] for e in shared))
assert shared, 'A and B must share a canonical entity for the bridge to exist'

q  = 'Marie Curie'
qv = emb([q])[0].tobytes()
nm = lambda hits: [doc_of[h['id']] for h in (hits or []) if h.get('id')]
hyb = nm(bdb.search(q, qv, columns=['content','id'], limit=8))
grf = nm(bdb.graph_search(q, qv, columns=['content'], limit=8, seed_n=2, hops=2))
print('hybrid:', hyb)
print('graph :', grf)
print(f"rank of B  hybrid={hyb.index('B')+1 if 'B' in hyb else '-'}  graph={grf.index('B')+1 if 'B' in grf else '-'}")

A entities: ['marie curie', 'paris', 'polonium']
B entities: ['alpha emission', 'polonium']
shared    : ['polonium']
hybrid: ['A', 'C', 'G', 'D', 'H', 'B', 'E', 'F']
graph : ['A', 'C', 'B', 'G', 'D', 'H', 'E', 'F']
rank of B  hybrid=6  graph=3


### Where this stops

No LLM is involved anywhere above, and for retrieval that is fine. What you do **not** get:
faithful typed relations over open-domain prose, a normalised relation vocabulary, and
readable community summaries — topic labels here are yake keyphrases, which are good enough
to filter and debug with but are not summaries. Those are the parts worth spending a model on,
and they can be done offline once (distil a label set / schema) rather than per document.